# Pass 7 — Preprocessing Ablation and Statistical Uncertainty

## Objective

Determine whether image preprocessing contributes additional classification benefit **after lesion localization**, while preserving the locked Pass 3B cohort split and the same VGG16 classifier protocol.

### Conditions

1. **Raw ROI** — original grayscale ultrasound image, lesion-localized using the BUSI mask.
2. **Min-Max ROI** — per-image Min-Max normalization followed by lesion localization.
3. **CLAHE ROI** — CLAHE applied to the original image followed by lesion localization.
4. **Min-Max + CLAHE ROI** — Min-Max normalization followed by CLAHE and lesion localization.

The primary comparison is among these four **lesion-localized** conditions. The previously evaluated whole-image model remains a contextual reference but is not retrained here.

### Statistical protocol

- Locked Pass 3B split: 455 train / 96 validation / 96 test.
- Same image membership across all four conditions.
- Same VGG16 architecture and training procedure.
- Same random seed.
- Augmentation applied only to training data.
- Validation/test data are unaugmented.
- Primary threshold: 0.50.
- Primary metrics: accuracy, balanced accuracy, sensitivity, specificity, precision, F1, ROC-AUC, PR-AUC.
- 10,000 image-level bootstrap replicates for 95% CIs.
- Test predictions are saved before bootstrap analysis.
- No test-set preprocessing or threshold tuning.


In [ ]:
# Imports and reproducibility
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = Path.cwd()
PASS3B_DIR = PROJECT_ROOT / "results" / "pass3b_vgg16"
SPLIT_CSV = PASS3B_DIR / "split" / "Pass_3B_locked_split.csv"

OUT_DIR = PROJECT_ROOT / "results" / "pass7_preprocessing_ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)
(OUT_DIR / "tables").mkdir(exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 40
THRESHOLD = 0.50
N_BOOT = 10_000

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("Split:", SPLIT_CSV)


In [ ]:
# Load and lock the Pass 3B split
assert SPLIT_CSV.exists(), f"Missing locked split: {SPLIT_CSV}"
split = pd.read_csv(SPLIT_CSV)

required = {"image_path", "split", "label"}
missing = required - set(split.columns)
assert not missing, f"Missing required columns: {missing}"

split["label"] = split["label"].astype(int)
assert set(split["label"].unique()) == {0, 1}

counts = split.groupby(["split", "label"]).size().unstack(fill_value=0)
print(counts)

expected = {
    ("train", 0): 319, ("train", 1): 136,
    ("validation", 0): 65, ("validation", 1): 31,
    ("test", 0): 65, ("test", 1): 31,
}
for key, value in expected.items():
    assert int(counts.loc[key[0], key[1]]) == value, (key, counts.loc[key[0], key[1]], value)

assert len(split) == 647
assert split.groupby("image_path").size().max() == 1

print("Locked cohort verified: 455 train / 96 validation / 96 test.")


## Image preprocessing definitions

To avoid ambiguity, the four conditions are reconstructed from the original image and the original BUSI lesion mask.

The lesion crop uses the same **15×15 elliptical morphological dilation** established in the Pass 3B raw-ROI pipeline. The dilation is used only to define the model input crop; the original lesion mask remains the anatomical annotation.

Preprocessing is performed **per image**, so no dataset-wide normalization statistics are estimated from the validation or test sets.


In [ ]:
# Locate original image and tumor mask paths robustly
def first_existing(row, candidates):
    for c in candidates:
        if c in row.index and pd.notna(row[c]):
            p = Path(str(row[c]))
            if p.exists():
                return p
            # allow paths relative to project root
            q = PROJECT_ROOT / p
            if q.exists():
                return q
    return None

# Pass 3B normally contains these paths. Resolve defensively if names differ.
IMAGE_COLS = ["image_path", "original_image_path", "source_image_path"]
MASK_COLS = ["tumor_mask_path", "mask_path", "original_mask_path"]

resolved = []
for _, row in split.iterrows():
    img = first_existing(row, IMAGE_COLS)
    mask = first_existing(row, MASK_COLS)
    if img is None or mask is None:
        raise FileNotFoundError(
            f"Could not resolve image/mask for row: {row.to_dict()}"
        )
    resolved.append((img, mask))

split["_resolved_image"] = [str(x[0]) for x in resolved]
split["_resolved_mask"] = [str(x[1]) for x in resolved]

assert all(Path(p).exists() for p in split["_resolved_image"])
assert all(Path(p).exists() for p in split["_resolved_mask"])
print("Resolved images and masks:", len(split))


In [ ]:
# Preprocessing functions
KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

def read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    return img

def read_mask(path, shape):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {path}")
    if mask.shape != shape:
        raise ValueError(f"Mask/image shape mismatch: {path}: {mask.shape} vs {shape}")
    return mask

def minmax01(img):
    x = img.astype(np.float32)
    lo, hi = float(x.min()), float(x.max())
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)

def clahe01(img01):
    # CLAHE operates on uint8; output is returned in [0, 1].
    u8 = np.clip(img01 * 255.0, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(u8).astype(np.float32) / 255.0

def preprocess_image(img, condition):
    if condition == "raw_roi":
        return img.astype(np.float32) / 255.0
    if condition == "minmax_roi":
        return minmax01(img)
    if condition == "clahe_roi":
        return clahe01(img.astype(np.float32) / 255.0)
    if condition == "minmax_clahe_roi":
        return clahe01(minmax01(img))
    raise ValueError(condition)

def extract_roi(image_path, mask_path, condition):
    img = read_gray(image_path)
    mask = read_mask(mask_path, img.shape)

    # Original BUSI lesion mask -> 15x15 elliptical dilation.
    dilated = cv2.dilate((mask > 0).astype(np.uint8), KERNEL, iterations=1)

    ys, xs = np.where(dilated > 0)
    if len(xs) == 0:
        raise ValueError(f"Empty lesion mask: {mask_path}")

    processed = preprocess_image(img, condition)
    localized = processed * (dilated.astype(np.float32))

    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    crop = localized[y0:y1+1, x0:x1+1]

    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    crop = np.clip(crop, 0.0, 1.0).astype(np.float32)

    # VGG16 expects 3 channels; replicate grayscale channel.
    return np.repeat(crop[..., None], 3, axis=-1)


In [ ]:
# Build all four datasets once and verify identical membership
CONDITIONS = ["raw_roi", "minmax_roi", "clahe_roi", "minmax_clahe_roi"]

X = {c: [] for c in CONDITIONS}
y = split["label"].to_numpy(dtype=np.int32)

for i, row in split.reset_index(drop=True).iterrows():
    for c in CONDITIONS:
        X[c].append(
            extract_roi(row["_resolved_image"], row["_resolved_mask"], c)
        )

for c in CONDITIONS:
    X[c] = np.stack(X[c]).astype(np.float32)
    assert X[c].shape == (647, 224, 224, 3)
    assert np.isfinite(X[c]).all()
    assert X[c].min() >= 0.0 and X[c].max() <= 1.0

# Verify that sample ordering is identical across all conditions.
for c in CONDITIONS[1:]:
    assert len(X[c]) == len(X["raw_roi"])

print({c: X[c].shape for c in CONDITIONS})


In [ ]:
# Exact split indices
train_idx = np.flatnonzero(split["split"].eq("train").to_numpy())
val_idx = np.flatnonzero(split["split"].eq("validation").to_numpy())
test_idx = np.flatnonzero(split["split"].eq("test").to_numpy())

assert len(train_idx) == 455
assert len(val_idx) == 96
assert len(test_idx) == 96
assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)


## Model protocol

The architecture matches the Pass 3B baseline:

- ImageNet VGG16 backbone, frozen.
- GlobalAveragePooling2D.
- Dense(256, ReLU).
- Dropout(0.5).
- Dense(1, sigmoid).
- Adam, learning rate 1e-4.
- Early stopping on validation ROC-AUC, patience 7.
- ReduceLROnPlateau on validation ROC-AUC.
- Training augmentation only.

Each condition gets an independently trained model, but **no condition is given access to the test labels during training or model selection**.


In [ ]:
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import VGG16

def build_model():
    backbone = VGG16(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    z = backbone(inputs, training=False)
    z = layers.GlobalAveragePooling2D()(z)
    z = layers.Dense(256, activation="relu")(z)
    z = layers.Dropout(0.5)(z)
    outputs = layers.Dense(1, activation="sigmoid")(z)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.BinaryAccuracy(name="accuracy")
        ],
    )
    return model

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.05, seed=SEED),
    layers.RandomZoom(0.10, seed=SEED),
], name="train_augmentation")


In [ ]:
def make_train_ds(x, labels, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((x, labels))
    ds = ds.shuffle(len(labels), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(lambda a, b: (augment(a, training=True), b),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def make_eval_ds(x, labels):
    return tf.data.Dataset.from_tensor_slices((x, labels)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def evaluate_predictions(y_true, prob, threshold=THRESHOLD):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    out = {
        "n": len(y_true),
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "brier_score": brier_score_loss(y_true, prob),
        "log_loss": log_loss(y_true, np.clip(prob, 1e-7, 1-1e-7)),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    return out


In [ ]:
# Train/evaluate all four conditions
all_metrics = []
all_predictions = []
histories = {}

for j, condition in enumerate(CONDITIONS):
    print(f"\n=== {condition} ===")

    # Reset graph and seeds for reproducibility.
    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_model()

    ckpt = OUT_DIR / "models" / f"vgg16_{condition}.keras"
    cb = [
        callbacks.ModelCheckpoint(
            str(ckpt), monitor="val_auc", mode="max",
            save_best_only=True, verbose=0
        ),
        callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=7,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.5,
            patience=3, min_lr=1e-7, verbose=1
        ),
    ]

    history = model.fit(
        make_train_ds(X[condition][train_idx], y[train_idx]),
        validation_data=make_eval_ds(X[condition][val_idx], y[val_idx]),
        epochs=EPOCHS,
        callbacks=cb,
        verbose=1
    )
    histories[condition] = history.history

    prob = model.predict(
        make_eval_ds(X[condition][test_idx], y[test_idx]),
        verbose=0
    ).reshape(-1)

    m = evaluate_predictions(y[test_idx], prob)
    m["condition"] = condition
    all_metrics.append(m)

    test_rows = split.iloc[test_idx].copy().reset_index(drop=True)
    pred_df = pd.DataFrame({
        "condition": condition,
        "image_path": test_rows["_resolved_image"].to_numpy(),
        "true_label": y[test_idx],
        "predicted_probability": prob,
        "predicted_label": (prob >= THRESHOLD).astype(int),
    })
    all_predictions.append(pred_df)

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True)

metrics_df.to_csv(OUT_DIR / "tables" / "pass7_ablation_metrics.csv", index=False)
predictions_df.to_csv(OUT_DIR / "tables" / "pass7_test_predictions.csv", index=False)

metrics_df


In [ ]:
# Bootstrap 95% CIs — image-level resampling within each condition
def bootstrap_metric_ci(y_true, prob, metric_name, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    n = len(y_true)
    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        pp = prob[idx]
        try:
            if metric_name == "accuracy":
                v = accuracy_score(yt, pp >= THRESHOLD)
            elif metric_name == "balanced_accuracy":
                v = balanced_accuracy_score(yt, pp >= THRESHOLD)
            elif metric_name == "precision":
                v = precision_score(yt, pp >= THRESHOLD, zero_division=0)
            elif metric_name == "sensitivity":
                v = recall_score(yt, pp >= THRESHOLD, zero_division=0)
            elif metric_name == "specificity":
                tn, fp, fn, tp = confusion_matrix(yt, pp >= THRESHOLD, labels=[0,1]).ravel()
                v = tn / (tn + fp) if (tn + fp) else np.nan
            elif metric_name == "f1":
                v = f1_score(yt, pp >= THRESHOLD, zero_division=0)
            elif metric_name == "roc_auc":
                v = roc_auc_score(yt, pp) if len(np.unique(yt)) == 2 else np.nan
            elif metric_name == "pr_auc":
                v = average_precision_score(yt, pp) if len(np.unique(yt)) == 2 else np.nan
            elif metric_name == "brier_score":
                v = brier_score_loss(yt, pp)
            elif metric_name == "log_loss":
                v = log_loss(yt, np.clip(pp, 1e-7, 1-1e-7))
            else:
                raise ValueError(metric_name)
            vals.append(v)
        except Exception:
            vals.append(np.nan)

    vals = np.asarray(vals, dtype=float)
    finite = vals[np.isfinite(vals)]
    return {
        "estimate": float(evaluate_predictions(y_true, prob)[metric_name]),
        "ci95_low": float(np.quantile(finite, 0.025)),
        "ci95_high": float(np.quantile(finite, 0.975)),
        "n_bootstrap_finite": int(len(finite)),
    }

BOOT_METRICS = [
    "accuracy", "balanced_accuracy", "precision", "sensitivity",
    "specificity", "f1", "roc_auc", "pr_auc", "brier_score", "log_loss"
]

boot_rows = []
for condition in CONDITIONS:
    d = predictions_df[predictions_df["condition"].eq(condition)].reset_index(drop=True)
    for metric in BOOT_METRICS:
        r = bootstrap_metric_ci(
            d["true_label"].to_numpy(),
            d["predicted_probability"].to_numpy(),
            metric
        )
        boot_rows.append({
            "condition": condition,
            "metric": metric,
            **r
        })

bootstrap_df = pd.DataFrame(boot_rows)
bootstrap_df.to_csv(OUT_DIR / "tables" / "pass7_bootstrap_ci.csv", index=False)
bootstrap_df


## Pairwise comparison of preprocessing conditions

The primary scientific comparison is **not** a ranking based on a single metric. Instead, report the observed metric differences and their bootstrap uncertainty.

The most relevant contrasts are:

- Raw ROI vs Min-Max ROI
- Raw ROI vs CLAHE ROI
- Raw ROI vs Min-Max + CLAHE ROI

A positive difference below means the first condition has the larger metric. These comparisons are descriptive and uncertainty-aware; they are not used to tune the model.


In [ ]:
# Paired bootstrap differences: same test cases resampled jointly
PAIRWISE = [
    ("raw_roi", "minmax_roi"),
    ("raw_roi", "clahe_roi"),
    ("raw_roi", "minmax_clahe_roi"),
]

def metric_value(y_true, prob, metric):
    return evaluate_predictions(y_true, prob)[metric]

pair_rows = []
rng = np.random.default_rng(SEED)

for a, b in PAIRWISE:
    da = predictions_df[predictions_df["condition"].eq(a)].sort_values("image_path").reset_index(drop=True)
    db = predictions_df[predictions_df["condition"].eq(b)].sort_values("image_path").reset_index(drop=True)

    assert np.array_equal(da["image_path"].to_numpy(), db["image_path"].to_numpy())
    assert np.array_equal(da["true_label"].to_numpy(), db["true_label"].to_numpy())

    ya = da["true_label"].to_numpy()
    pa = da["predicted_probability"].to_numpy()
    pb = db["predicted_probability"].to_numpy()
    n = len(ya)

    for metric in BOOT_METRICS:
        vals = []
        for _ in range(N_BOOT):
            idx = rng.integers(0, n, size=n)
            try:
                va = metric_value(ya[idx], pa[idx], metric)
                vb = metric_value(ya[idx], pb[idx], metric)
                vals.append(va - vb)
            except Exception:
                vals.append(np.nan)

        vals = np.asarray(vals)
        vals = vals[np.isfinite(vals)]
        observed = metric_value(ya, pa, metric) - metric_value(ya, pb, metric)

        pair_rows.append({
            "condition_a": a,
            "condition_b": b,
            "metric": metric,
            "observed_difference_a_minus_b": observed,
            "ci95_low": np.quantile(vals, 0.025),
            "ci95_high": np.quantile(vals, 0.975),
            "n_bootstrap_finite": len(vals),
        })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df.to_csv(OUT_DIR / "tables" / "pass7_pairwise_bootstrap_differences.csv", index=False)
pairwise_df


In [ ]:
# Confusion matrices and calibration summaries
cm_rows = []
cal_rows = []

for condition in CONDITIONS:
    d = predictions_df[predictions_df["condition"].eq(condition)].reset_index(drop=True)
    y_true = d["true_label"].to_numpy()
    prob = d["predicted_probability"].to_numpy()
    pred = d["predicted_label"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    cm_rows.append({
        "condition": condition, "tn": tn, "fp": fp, "fn": fn, "tp": tp
    })

    cal_rows.append({
        "condition": condition,
        "n": len(y_true),
        "brier_score": brier_score_loss(y_true, prob),
        "log_loss": log_loss(y_true, np.clip(prob, 1e-7, 1-1e-7))
    })

pd.DataFrame(cm_rows).to_csv(
    OUT_DIR / "tables" / "pass7_confusion_matrices.csv", index=False
)
pd.DataFrame(cal_rows).to_csv(
    OUT_DIR / "tables" / "pass7_calibration_summary.csv", index=False
)

print("Saved confusion matrices and calibration summaries.")


In [ ]:
# Save training histories
with open(OUT_DIR / "tables" / "pass7_training_histories.json", "w") as f:
    json.dump(histories, f)

manifest = {
    "seed": SEED,
    "n_total": int(len(split)),
    "n_train": int(len(train_idx)),
    "n_validation": int(len(val_idx)),
    "n_test": int(len(test_idx)),
    "conditions": CONDITIONS,
    "threshold": THRESHOLD,
    "bootstrap_replicates": N_BOOT,
    "roi_kernel": "15x15 elliptical morphological dilation",
    "model": {
        "backbone": "VGG16 ImageNet pretrained, frozen",
        "head": "GlobalAveragePooling2D -> Dense(256,relu) -> Dropout(0.5) -> Dense(1,sigmoid)",
        "optimizer": "Adam",
        "learning_rate": 1e-4,
        "early_stopping_monitor": "val_auc",
        "early_stopping_patience": 7
    },
    "test_tuning": "none",
    "external_data_used_for_tuning": False
}
with open(OUT_DIR / "pass7_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))


## Publication interpretation guardrails

Pass 7 should answer whether preprocessing adds measurable benefit **within the same lesion-localized classification task**.

Do not claim that a preprocessing method is clinically superior based on one metric or one point estimate. The primary evidence should be:

1. absolute performance with 95% CIs;
2. paired test-set differences with 95% bootstrap CIs;
3. consistency across discrimination and threshold-dependent metrics;
4. calibration changes;
5. reproducibility under the locked split.

If a CI for a pairwise difference contains zero, describe the observed difference as **uncertain under this bootstrap analysis**, rather than claiming equivalence or superiority.
